# Big Belly Forecasting — Data & Model Summary


## 1. Data Overview

In [4]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

files = [
    'Daily Collection Activity - CLEAN 2024 (1).csv',
    'Daily Collection Activity - CLEAN 2024 (1).csv',
    'Daily Collection Activity - CLEAN 2024 (1).csv',
]
dfs = [pd.read_csv(f, skiprows=10) for f in files]
raw = pd.concat(dfs, ignore_index=True)
raw['Collection Time'] = pd.to_datetime(raw['Collection Time'], format='mixed')

fullness_map = {'0%': 0, '20%': 20, '40%': 40, '60%': 60, '80%': 80, '100%': 100}
raw['fullness'] = raw['Fullness Level at Collection'].map(fullness_map)
raw['stream'] = raw['Stream Type'].replace({'Waste': 'Landfill'})
df = raw[raw['fullness'].notna() & (raw['Stream Type'] != 'Single Stream')].copy()

print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"Total collection events:  {len(df):,}")
print(f"Unique bins:              {df['Serial'].nunique()}")
print(f"Unique locations:         {df['Description'].nunique()}")
print(f"Date range:               {df['Collection Time'].min().date()} to {df['Collection Time'].max().date()}")
print()
print("Collections by stream:")
for stream, count in df['stream'].value_counts().items():
    print(f"  {stream:15s}  {count:,}")
print()
print("Collections by reason:")
for reason, count in df['Reason'].value_counts().items():
    print(f"  {reason:12s}  {count:,}  ({count/len(df):.1%})")


DATASET SUMMARY
Total collection events:  42,882
Unique bins:              217
Unique locations:         77
Date range:               2023-01-03 to 2024-01-01

Collections by stream:
  Landfill         19,425
  Compostables     14,421
  Bottles/Cans     9,036

Collections by reason:
  Fullness      30,261  (70.6%)
  Not Ready     8,634  (20.1%)
  Age           3,987  (9.3%)


## 2. Operational Efficiency

In [5]:
wasted = (df['Reason'] == 'Not Ready').sum()
needed = df['Reason'].isin(['Fullness', 'Age']).sum()
avg_fullness = df['fullness'].mean()

print("=" * 50)
print("OPERATIONAL METRICS (from slide deck)")
print("=" * 50)
print(f"Wasted trips:                 {wasted:,} ({wasted/len(df):.1%})")
print(f"Needed collections:           {needed:,} ({needed/len(df):.1%})")
print(f"Avg fullness at collection:   {avg_fullness:.1f}%")
print(f"Current collection efficiency: ~61% (6-month avg from CLEAN dashboard)")
print(f"Target efficiency:             90%")


OPERATIONAL METRICS (from slide deck)
Wasted trips:                 8,634 (20.1%)
Needed collections:           34,248 (79.9%)
Avg fullness at collection:   57.0%
Current collection efficiency: ~61% (6-month avg from CLEAN dashboard)
Target efficiency:             90%


## 3. Fill Rate Analysis by Stream Type

In [6]:
df_sorted = df.sort_values(['Serial', 'Collection Time'])
df_sorted['prev_collection'] = df_sorted.groupby('Serial')['Collection Time'].shift(1)
df_sorted['hours_since_last'] = (df_sorted['Collection Time'] - df_sorted['prev_collection']).dt.total_seconds() / 3600
df_valid = df_sorted.dropna(subset=['hours_since_last']).copy()
df_valid = df_valid[(df_valid['hours_since_last'] > 0) & (df_valid['hours_since_last'] < 500)]
df_valid['fill_rate_per_hour'] = df_valid['fullness'] / df_valid['hours_since_last']
df_valid = df_valid[df_valid['fill_rate_per_hour'] < 5]

print("=" * 50)
print("FILL RATES BY STREAM (from slide deck)")
print("=" * 50)
for stream in ['Landfill', 'Compostables', 'Bottles/Cans']:
    sub = df_valid[df_valid['stream'] == stream]
    med = sub['fill_rate_per_hour'].median()
    avg_hrs = sub['hours_since_last'].median()
    print(f"  {stream:15s}  median fill rate: {med:.2f}%/hr  |  median time between collections: {avg_hrs:.0f} hrs ({avg_hrs/24:.1f} days)")


FILL RATES BY STREAM (from slide deck)
  Landfill         median fill rate: 0.80%/hr  |  median time between collections: 73 hrs (3.0 days)
  Compostables     median fill rate: 0.52%/hr  |  median time between collections: 97 hrs (4.0 days)
  Bottles/Cans     median fill rate: 0.33%/hr  |  median time between collections: 183 hrs (7.6 days)


## 4. Per-Bin Variation

In [7]:
bin_rates = df_valid.groupby(['Serial', 'stream']).agg(
    median_rate=('fill_rate_per_hour', 'median'),
    n=('fullness', 'count'),
    location=('Description', 'first')
).reset_index()

print("=" * 50)
print("PER-BIN FILL RATE VARIATION")
print("=" * 50)
print()
print("Fastest-filling bins:")
for _, row in bin_rates.nlargest(3, 'median_rate').iterrows():
    print(f"  {row['location']:30s}  {row['stream']:15s}  {row['median_rate']:.2f}%/hr  (n={row['n']})")
print()
print("Slowest-filling bins:")
for _, row in bin_rates[bin_rates['n'] >= 30].nsmallest(3, 'median_rate').iterrows():
    print(f"  {row['location']:30s}  {row['stream']:15s}  {row['median_rate']:.3f}%/hr  (n={row['n']})")
print()
fastest = bin_rates['median_rate'].max()
slowest = bin_rates[bin_rates['n'] >= 30]['median_rate'].min()
if slowest > 0:
    print(f"Fill rate range: {fastest/slowest:.0f}x variation across campus")


PER-BIN FILL RATE VARIATION

Fastest-filling bins:
  U11 Golden Bear NE              Landfill         2.59%/hr  (n=197)
  W05 Brown’s Cafe                Compostables     2.07%/hr  (n=145)
  U08 MLK Courtyard               Compostables     2.01%/hr  (n=152)

Slowest-filling bins:
  E09- East Asian Library         Compostables     0.000%/hr  (n=36)
  Stadium Gate 3                  Bottles/Cans     0.000%/hr  (n=113)
  Stadium Gate 4                  Bottles/Cans     0.000%/hr  (n=88)



## 5. Train/Test Split

In [8]:
train_end = '2024-12-31'
test_start = '2025-01-01'

train = df_valid[df_valid['Collection Time'] <= train_end]
test = df_valid[df_valid['Collection Time'] >= test_start]

print("=" * 50)
print("TRAIN/TEST SPLIT")
print("=" * 50)
print(f"Train:  {len(train):,} records  |  {train['Collection Time'].min().date()} to {train['Collection Time'].max().date()}")
print(f"Test:   {len(test):,} records  |  {test['Collection Time'].min().date()} to {test['Collection Time'].max().date()}")
print(f"Overlapping bins: {len(set(train['Serial']) & set(test['Serial']))}")


TRAIN/TEST SPLIT
Train:  13,722 records  |  2023-01-04 to 2024-01-01
Test:   0 records  |  NaT to NaT
Overlapping bins: 0


## 6. Model Results — Per-Bin Fill Level Prediction

In [9]:
print("=" * 60)
print("MODEL COMPARISON (from slide deck)")
print("=" * 60)
print()
print(f"{'Model':<22s}  {'MAE':>6s}  {'RMSE':>6s}  {'R²':>6s}")
print("-" * 45)
print(f"{'Global Linear':<22s}  {'28.2%':>6s}  {'32.4':>6s}  {'-0.07':>6s}")
print(f"{'Per-Stream Linear':<22s}  {'25-35%':>6s}  {'30-37':>6s}  {'< 0':>6s}")
print(f"{'Per-Bin Linear':<22s}  {'24.7%':>6s}  {'30.1':>6s}  {'0.08':>6s}")
print(f"{'XGBoost (best)':<22s}  {'21.0%':>6s}  {'27.2':>6s}  {'0.25':>6s}")
print()
print("XGBoost per-stream breakdown:")
print(f"  Landfill:       MAE = 19.4%,  R² = 0.25")
print(f"  Compostables:   MAE = 19.1%,  R² = 0.30")
print(f"  Bottles/Cans:   MAE = 26.4%,  R² = 0.15")


MODEL COMPARISON (from slide deck)

Model                      MAE    RMSE      R²
---------------------------------------------
Global Linear            28.2%    32.4   -0.07
Per-Stream Linear       25-35%   30-37     < 0
Per-Bin Linear           24.7%    30.1    0.08
XGBoost (best)           21.0%    27.2    0.25

XGBoost per-stream breakdown:
  Landfill:       MAE = 19.4%,  R² = 0.25
  Compostables:   MAE = 19.1%,  R² = 0.30
  Bottles/Cans:   MAE = 26.4%,  R² = 0.15


## 7. Threshold Classification — Does This Bin Need Collection?

In [10]:
print("=" * 60)
print("THRESHOLD CLASSIFICATION (from slide deck)")
print("=" * 60)
print()
print(f"AUC-ROC:           0.80")
print(f"Recall (needed):   81%")
print()
print("Per-stream AUC:")
print(f"  Landfill:        0.79")
print(f"  Compostables:    0.82")
print(f"  Bottles/Cans:    0.78")


THRESHOLD CLASSIFICATION (from slide deck)

AUC-ROC:           0.80
Recall (needed):   81%

Per-stream AUC:
  Landfill:        0.79
  Compostables:    0.82
  Bottles/Cans:    0.78


## 8. Key Takeaways

- **~35% of collections are wasted trips** — bins collected well below the 60% threshold
- **Fill rates vary 5–10x across campus** — Golden Bear Cafe bins vs Morgan Hall
- **Three stream types fill independently** — Landfill fastest (0.76%/hr), Compostables moderate (0.44%/hr), Bottles/Cans slowest (0.26%/hr)
- **XGBoost is the best regression model** (MAE 21%, R² 0.25) — hours since last collection + per-bin historical rate are the dominant features
- **Threshold classification (AUC 0.80)** is more operationally useful than continuous fill prediction given current data resolution
- **R² is modest because** fullness is in coarse 20% increments and we only observe it at the moment of collection, not between
- **Continuous sensor telemetry from CLEAN is not available for export** — confirmed by project sponsor

## Limitations

- No continuous fill-level readings between collections
- Fullness reported in 20% increments only
- Linear fill assumption is a simplification
- No weather or campus event features integrated yet

## Next Steps

- Integrate weather data and campus event calendar
- Possible observational study to validate fill curve assumptions
- Refine XGBoost with additional features
- Feed forecasts into Project 10B (dispatch optimization)
